# Defaults settings

In [1]:
import pandas as pd
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
import os
import warnings
warnings.filterwarnings('ignore')

In [52]:
MODEL_PATH = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"  # deepseek-ai/DeepSeek-R1-Distill-Qwen-32B    meta-llama/Llama-3.1-8B-Instruct
TP_SIZE = 1  # 你的GPU数量

print(f" 正在加载 Tokenizer 和模型 {MODEL_PATH} ... ")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

# 初始化 vLLM 引擎
llm = LLM(
    model=MODEL_PATH,
    tensor_parallel_size=TP_SIZE,
    dtype="bfloat16",
    max_model_len=1024,            # 上下文截断
    gpu_memory_utilization=0.85,  # 显存预占比例
    enforce_eager=True            # 关闭 CUDA Graph 防止 OOM
)

 正在加载 Tokenizer 和模型 deepseek-ai/DeepSeek-R1-Distill-Llama-8B ... 


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

INFO 05-06 15:53:45 [utils.py:233] non-default args: {'dtype': 'bfloat16', 'max_model_len': 1024, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B'}


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

INFO 05-06 15:53:47 [model.py:549] Resolved architecture: LlamaForCausalLM
INFO 05-06 15:53:47 [model.py:1678] Using max model len 1024
WARNING 05-06 15:53:47 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-06 15:53:47 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-06 15:53:47 [vllm.py:1025] Cudagraph is disabled under eager mode


generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

(EngineCore pid=860466) INFO 05-06 15:53:48 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='deepseek-ai/DeepSeek-R1-Distill-Llama-8B', speculative_config=None, tokenizer='deepseek-ai/DeepSeek-R1-Distill-Llama-8B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=1024, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_

(EngineCore pid=860466) Process EngineCore:
(EngineCore pid=860466) Traceback (most recent call last):
(EngineCore pid=860466)   File "/home/shsi/.conda/envs/promptllm/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=860466)     self.run()
(EngineCore pid=860466)   File "/home/shsi/.conda/envs/promptllm/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore pid=860466)     self._target(*self._args, **self._kwargs)
(EngineCore pid=860466)   File "/home/shsi/.conda/envs/promptllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1112, in run_engine_core
(EngineCore pid=860466)     raise e
(EngineCore pid=860466)   File "/home/shsi/.conda/envs/promptllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1082, in run_engine_core
(EngineCore pid=860466)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=860466)   File "/home/shsi/.conda/envs/promptllm/lib/python3.10/site-packages/vllm/

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}

In [8]:
sampling_params = SamplingParams(
    temperature=0., 
    max_tokens=150, 
    # top_p=0.95,
    # stop=["\n\n", "User:"]
)

In [46]:
# 1. 手动挑选 10 个具有代表性的例子（包含正确答案的 Ground Truth 用于自动算分）
test_cases = [
    {"Sub": "car", "Rel": "on", "Obj": "street", "Expected": "Yes"},

    {"Sub": "person", "Rel": "wearing", "Obj": "hat", "Expected": "Yes"},
    {"Sub": "boat", "Rel": "flying in", "Obj": "room", "Expected": "No"},
    {"Sub": "cat", "Rel": "sitting on", "Obj": "table", "Expected": "Yes"},
    {"Sub": "pizza", "Rel": "eating", "Obj": "man", "Expected": "No"},
    {"Sub": "train", "Rel": "parked on", "Obj": "track", "Expected": "Yes"},
    {"Sub": "tree", "Rel": "riding", "Obj": "bike", "Expected": "No"},
    {"Sub": "dog", "Rel": "holding", "Obj": "branch", "Expected": "Yes"},
    {"Sub": "flower", "Rel": "growing on", "Obj": "rock", "Expected": "Yes"},

    {"Sub": "logo", "Rel": "of", "Obj": "banana", "Expected": "Yes"},

    {"Sub": "person", "Rel": "along", "Obj": "towel", "Expected": "No"},
    {"Sub": "vase", "Rel": "sitting on", "Obj": "airplane", "Expected": "No"},
    {"Sub": "bag", "Rel": "across", "Obj": "street", "Expected": "No"},
    {"Sub": "leg", "Rel": "of", "Obj": "branch", "Expected": "No"},
    {"Sub": "laptop", "Rel": "of", "Obj": "screen", "Expected": "No"},
    {"Sub": "girl", "Rel": "attached to", "Obj": "book", "Expected": "No"},
    {"Sub": "man", "Rel": "eating", "Obj": "surfboard", "Expected": "No"},
    {"Sub": "man", "Rel": "says", "Obj": "snowboard", "Expected": "No"},


    # --- 陷阱 1：主客体颠倒 / 逆向逻辑 (测试模型是否只看词语相关性，不看语序) ---
    {"Sub": "plant", "Rel": "eating", "Obj": "cow", "Expected": "No"},       # 牛吃草，草不能吃牛
    {"Sub": "car", "Rel": "riding", "Obj": "boy", "Expected": "No"},         # 男孩骑车/乘车，车不能骑男孩
    {"Sub": "food", "Rel": "eating", "Obj": "girl", "Expected": "No"},       # 食物不能吃人
    {"Sub": "number", "Rel": "says", "Obj": "sign", "Expected": "No"},       # 标牌上写着数字，数字不能"写着"标牌

    # --- 陷阱 2：物理尺度与承重常识 (测试模型是否考虑物体的实际大小和重量) ---
    {"Sub": "elephant", "Rel": "sitting on", "Obj": "chair", "Expected": "No"},  # 椅子承受不了大象
    {"Sub": "bird", "Rel": "carrying", "Obj": "horse", "Expected": "No"},        # 鸟叼不动马
    {"Sub": "cup", "Rel": "holding", "Obj": "elephant", "Expected": "No"},       # 杯子装不下大象
    {"Sub": "airplane", "Rel": "parked on", "Obj": "roof", "Expected": "No"},    # 真实世界的飞机不会停在普通屋顶上
    {"Sub": "car", "Rel": "in", "Obj": "box", "Expected": "No"},                 # 排除玩具特例，真车装不进普通纸箱

    # --- 陷阱 3：过度拟人化 / 动画片隐喻 (测试模型是否把童话当成现实) ---
    {"Sub": "motorcycle", "Rel": "wearing", "Obj": "helmet", "Expected": "No"},  # 摩托车不戴头盔，骑手才戴
    {"Sub": "shoe", "Rel": "walking on", "Obj": "sidewalk", "Expected": "No"},   # 鞋子本身不会走，必须是人穿着
    {"Sub": "dog", "Rel": "using", "Obj": "laptop", "Expected": "No"},           # 狗不具备使用电脑的物理能力
    {"Sub": "cat", "Rel": "wearing", "Obj": "boot", "Expected": "No"},           # 穿靴子的猫是童话，现实中视觉不合理
    {"Sub": "flower", "Rel": "holding", "Obj": "cup", "Expected": "No"},         # 花没有手，不能拿杯子
    {"Sub": "cabinet", "Rel": "wearing", "Obj": "coat", "Expected": "No"},       # 柜子不能“穿”外套（只能是里面挂着/装有）
    {"Sub": "bench", "Rel": "watching", "Obj": "bird", "Expected": "No"},        # 长椅没有视觉，不能 watching

    # --- 陷阱 4：生物特征错乱 / 机器与生物乱接 (测试解剖学常识) ---
    {"Sub": "wing", "Rel": "part of", "Obj": "dog", "Expected": "No"},           # 狗没有翅膀
    {"Sub": "leaf", "Rel": "part of", "Obj": "bird", "Expected": "No"},          # 鸟不长叶子
    {"Sub": "leg", "Rel": "attached to", "Obj": "wheel", "Expected": "No"},      # 真实的肉腿不会长在轮子上
    {"Sub": "wheel", "Rel": "belonging to", "Obj": "horse", "Expected": "No"},   # 马没有轮子
    {"Sub": "cow", "Rel": "sitting on", "Obj": "fence", "Expected": "No"},       # 牛的骨骼结构无法像人一样“坐”在篱笆上

    # --- 陷阱 5：空间介词混淆及极端特例 (讨论过的刁钻案例) ---
    {"Sub": "train", "Rel": "parked on", "Obj": "street", "Expected": "No"},     # 火车停在铁轨上，停在街上是极端特例
    {"Sub": "boat", "Rel": "parked on", "Obj": "tree", "Expected": "No"},        # 船不会停在树上
    {"Sub": "umbrella", "Rel": "flying in", "Obj": "room", "Expected": "No"},    # 雨伞在室内飞行（魔法/风暴极端情况）
    {"Sub": "logo", "Rel": "painted on", "Obj": "horse", "Expected": "No"},      # 活体马身上直接画Logo属于极端特例

    # --- 迷惑组：看似荒谬但物理现实完全成立的 Hard Positives (测试模型是否敢说 Yes) ---
    {"Sub": "bird", "Rel": "standing on", "Obj": "cow", "Expected": "Yes"},      # 牛背鹭等鸟类停在牛背上很常见
    {"Sub": "plant", "Rel": "growing on", "Obj": "building", "Expected": "Yes"}, # 爬山虎、苔藓长在建筑上
    {"Sub": "sign", "Rel": "says", "Obj": "number", "Expected": "Yes"},          # 标牌上写着限速等数字
    {"Sub": "cat", "Rel": "looking at", "Obj": "book", "Expected": "Yes"},       # 猫虽然不认字，但物理上盯着书看完全可能
    {"Sub": "dog", "Rel": "covered in", "Obj": "snow", "Expected": "Yes"},       # 狗在雪地玩耍身上覆满雪
    {"Sub": "tree", "Rel": "growing on", "Obj": "rock", "Expected": "Yes"},      # 树从岩石缝隙中长出来
    {"Sub": "bag", "Rel": "hanging from", "Obj": "branch", "Expected": "Yes"}    # 包挂在树枝上，视觉和物理上完全说得通
]

# 2. 匹配你原代码中的问题生成逻辑
def format_question(sub, rel, obj):
    sub_f = f'a group of {sub}' if sub in ['men', 'people'] else f'a {sub}'
    obj_f = f'a group of {obj}' if obj in ['men', 'people'] else f'a {obj}'
    
    rel_formatted = rel
    if rel != 'and':
        if rel == 'wears': rel_formatted = 'wearing'
        elif rel == 'says': rel_formatted = 'saying'
        elif rel == 'has': rel_formatted = 'have'
        
        rel_f = 'have' if rel_formatted == 'have' else f'be {rel_formatted}'
        return f"Can {sub_f} {rel_f} {obj_f}?"
    else:
        return f"Can {sub_f} {rel} {obj_f} appear together?"

# 3. 定义测试函数
def run_ablation(system_prompt_content, few_shots=[]):
    prompts = []
    
    # 构建所有问题的 Prompt
    for case in test_cases:
        question = format_question(case['Sub'], case['Rel'], case['Obj'])
        
        messages = [{"role": "system", "content": system_prompt_content}]
        messages.extend(few_shots)
        messages.append({"role": "user", "content": question})
        
        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prompts.append(prompt_text)
        
    # vLLM 批量推理 (关闭 tqdm 防止 Jupyter 输出过长)
    outputs = llm.generate(prompts, sampling_params, use_tqdm=False)
    
    # 结果解析与统计
    results = []
    for case, out in zip(test_cases, outputs):
        answer = out.outputs[0].text.strip()
        # 简单判定：如果模型输出包含预期答案（忽略大小写）则判对
        is_correct = case['Expected'].lower() in answer.lower()
        results.append({
            "Question": format_question(case['Sub'], case['Rel'], case['Obj']),
            "Expected": case['Expected'],
            "Model_Output": answer,
            "Correct": is_correct
        })
        
    df_results = pd.DataFrame(results)
    accuracy = df_results['Correct'].mean() * 100
    print(f"\n=> 测试完成! 在 {len(df_results['Correct'])}个样本上的准确率为: {accuracy:.1f}%\n")
    return df_results

# Exp 1: Original prompts (v4)

In [47]:
prompt_v1 = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
Interpret this strictly as a statement about physical and commonsense plausibility in the real world.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
Respond with a Yes or No if both commonsense and visually plausible, without additional explainations."""

shots_v1 = [
    {"role": "user", "content": "Can a car be driving on a road?"},
    {"role": "assistant", "content": "Yes"},
    {"role": "user", "content": "Can an animal be eating an apple?"},
    {"role": "assistant", "content": "Yes"},
    {"role": "user", "content": "Can a bird be under a mountain?"},
    {"role": "assistant", "content": "No"},
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "No"}
]

print("=== 实验 1: 原始长 Prompt + 4-shot ===")
df_res1 = run_ablation(prompt_v1, shots_v1)
display(df_res1) # 

=== 实验 1: 原始长 Prompt + 4-shot ===

=> 测试完成! 在 50个样本上的准确率为: 76.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,Yes,True
1,Can a person be wearing a hat?,Yes,Yes,True
2,Can a boat be flying in a room?,No,No,True
3,Can a cat be sitting on a table?,Yes,Yes,True
4,Can a pizza be eating a man?,No,No,True
5,Can a train be parked on a track?,Yes,Yes,True
6,Can a tree be riding a bike?,No,No,True
7,Can a dog be holding a branch?,Yes,Yes,True
8,Can a flower be growing on a rock?,Yes,Yes,True
9,Can a logo be of a banana?,Yes,Yes,True


In [48]:
# 1. 重新配置采样参数（增加长度，temperature可以为0确保单次推理最稳）
sampling_params_cot = SamplingParams(temperature=0, max_tokens=256)
pd.set_option('display.max_colwidth', None)
# 2. 定义 CoT Prompt
prompt_cot = """You are a commonsense reasoning assistant. 
For each question, analyze the physical and visual plausibility step by step.
Consider if the interaction is possible in the real world and could appear in a real photo.
End your response with 'Final Answer: Yes' or 'Final Answer: No'."""

shots_cot = [
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "Reasoning: 1) An arm is a biological part of a human body. 2) In the real world, arms cannot detach and fly independently. 3) This would only happen in CGI or fantasy. \nFinal Answer: No"},
    {"role": "user", "content": "Can a car be driving on a road?"},
    {"role": "assistant", "content": "Reasoning: 1) Cars are vehicles designed for roads. 2) This is a standard real-world activity. 3) It is visually plausible. \nFinal Answer: Yes"}
]

# 3. 运行测试（复用你的 run_ablation 函数，但传入新的采样参数）
# 注意：原 run_ablation 内部使用了 sampling_params，我们需要临时替换它
import copy
original_params = sampling_params
sampling_params = sampling_params_cot

print("=== 实验 2: Chain-of-Thought (CoT) ===")
df_res2 = run_ablation(prompt_cot, shots_cot)
display(df_res2)

# 还原参数
sampling_params = original_params

=== 实验 2: Chain-of-Thought (CoT) ===

=> 测试完成! 在 50个样本上的准确率为: 86.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,"Reasoning: 1) Cars are designed for roads, and streets are a type of road. 2) This is a common real-world scenario. 3) It is visually plausible. \nFinal Answer: Yes",True
1,Can a person be wearing a hat?,Yes,"Reasoning: 1) Humans wear clothing and accessories, including hats. 2) This is a common and everyday activity. 3) It is visually plausible. \nFinal Answer: Yes",True
2,Can a boat be flying in a room?,No,"Reasoning: 1) Boats are designed for water, not air. 2) They do not have the necessary aerodynamics to fly. 3) It would be difficult to create a visually plausible image of a boat flying in a room, as it would defy the laws of physics and gravity. \nFinal Answer: No",True
3,Can a cat be sitting on a table?,Yes,Reasoning: 1) Cats are living creatures that can move around. 2) Tables are common household objects. 3) It is a common and visually plausible scenario. \nFinal Answer: Yes,True
4,Can a pizza be eating a man?,No,"Reasoning: 1) A pizza is a food item, inanimate. 2) Eating is an action typically performed by living beings. 3) It is not physically possible for a pizza to consume a human. \nFinal Answer: No",True
5,Can a train be parked on a track?,Yes,Reasoning: 1) Trains are designed to run on tracks. 2) Parking a train on a track is a common real-world scenario. 3) It is visually plausible. \nFinal Answer: Yes,True
6,Can a tree be riding a bike?,No,Reasoning: 1) Trees are stationary objects and do not have the ability to move on their own. 2) Bicycles require a rider to operate. 3) It is not physically possible for a tree to ride a bike. \nFinal Answer: No,True
7,Can a dog be holding a branch?,Yes,Reasoning: 1) Dogs are living creatures with physical capabilities. 2) They can pick up objects in their mouths. 3) It is visually plausible for a dog to hold a branch in its mouth. \nFinal Answer: Yes,True
8,Can a flower be growing on a rock?,Yes,"Reasoning: 1) Flowers can grow in various environments, including rocky areas. 2) Plants can adapt to grow in small crevices or cracks on rocks. 3) This is a common occurrence in nature. \nFinal Answer: Yes",True
9,Can a logo be of a banana?,Yes,Reasoning: 1) Logos are graphical representations. 2) A banana is a real-world object. 3) It is visually plausible for a logo to be designed in the shape of a banana. \nFinal Answer: Yes,True


In [49]:
# 1. 重新配置采样参数（增加长度，temperature可以为0确保单次推理最稳）
sampling_params_cot = SamplingParams(temperature=0, max_tokens=256)
pd.set_option('display.max_colwidth', None)
# 2. 定义 CoT Prompt
prompt_cot = """You are a commonsense reasoning assistant. 
For each question, analyze the physical and visual plausibility step by step.
Consider if the interaction is possible in the real world and could appear in a real photo.
Strictly evaluate the spatial relationship defined by the preposition.
Do not substitute the relationship with a more plausible one (e.g., do not treat 'on' as 'in', or 'along' as 'with').
Do not invent stories or specific contexts to justify the situation. 
If it requires a 'dropped' or 'accident' scenario to be possible, respond with No.
Focus on the physical geometry. 
Focus on scale, volume, and physical orientation.
Can a human body physically align 'along' a towel's length? If the physical orientation is nonsensical, respond with No.


End your response with 'Final Answer: Yes' or 'Final Answer: No'."""

shots_cot = [
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "Reasoning: 1) An arm is a biological part of a human body. 2) In the real world, arms cannot detach and fly independently. 3) This would only happen in CGI or fantasy. \nFinal Answer: No"},
    {"role": "user", "content": "Can a car be driving on a road?"},
    {"role": "assistant", "content": "Reasoning: 1) Cars are vehicles designed for roads. 2) This is a standard real-world activity. 3) It is visually plausible. \nFinal Answer: Yes"},
    {"role": "user", "content": "Can a house be in a car?"},
    {"role": "assistant", "content": "Reasoning: 1) A house is significantly larger than a car. 2) A car cannot physically contain a house. 3) This violates physical scale and spatial logic. \nFinal Answer: No"},

]

# 3. 运行测试（复用你的 run_ablation 函数，但传入新的采样参数）
# 注意：原 run_ablation 内部使用了 sampling_params，我们需要临时替换它
import copy
original_params = sampling_params
sampling_params = sampling_params_cot

print("=== 实验 2: Chain-of-Thought (CoT) ===")
df_res2 = run_ablation(prompt_cot, shots_cot)
display(df_res2)

# 还原参数
sampling_params = original_params

=== 实验 2: Chain-of-Thought (CoT) ===

=> 测试完成! 在 50个样本上的准确率为: 92.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,Reasoning: 1) Cars are designed to be on roads. 2) Streets are a type of road. 3) This is a common and visually plausible scenario. \nFinal Answer: Yes,True
1,Can a person be wearing a hat?,Yes,Reasoning: 1) Humans wear hats as part of their attire. 2) Hats are designed to fit on a person's head. 3) This is a common and visually plausible scenario. \nFinal Answer: Yes,True
2,Can a boat be flying in a room?,No,"Reasoning: 1) Boats are designed for water, not air. 2) They do not have the physical properties to defy gravity. 3) It is not visually plausible in a real-world setting. \nFinal Answer: No",True
3,Can a cat be sitting on a table?,Yes,Reasoning: 1) Cats are small animals. 2) Tables are designed to support objects of various sizes. 3) It is visually plausible for a cat to sit on a table. \nFinal Answer: Yes,True
4,Can a pizza be eating a man?,No,Reasoning: 1) A pizza is an inanimate object. 2) It cannot consume or eat a human being. 3) This is a nonsensical and impossible scenario. \nFinal Answer: No,True
5,Can a train be parked on a track?,Yes,Reasoning: 1) Trains are designed to run on tracks. 2) Parking a train on a track is a common real-world scenario. 3) It is visually plausible. \nFinal Answer: Yes,True
6,Can a tree be riding a bike?,No,Reasoning: 1) Trees are stationary objects. 2) They do not have the ability to operate a bike. 3) A tree cannot physically sit on a bike seat or hold onto handlebars. \nFinal Answer: No,True
7,Can a dog be holding a branch?,Yes,"Reasoning: 1) Dogs have hands or paws, which can grasp objects. 2) Branches are objects that can be held. 3) This is a common real-world scenario. \nFinal Answer: Yes",True
8,Can a flower be growing on a rock?,Yes,"Reasoning: 1) Plants can grow in various environments, including rocks. 2) Roots can anchor into small crevices or cracks on rocks. 3) This is a visually plausible scenario. \nFinal Answer: Yes",True
9,Can a logo be of a banana?,Yes,Reasoning: 1) A logo is a visual representation. 2) A banana is an object that can be depicted in a logo. 3) This is a common and visually plausible scenario. \nFinal Answer: Yes,True


In [50]:
# 1. 重新配置采样参数（增加长度，temperature可以为0确保单次推理最稳）
sampling_params_cot = SamplingParams(temperature=0, max_tokens=256)
pd.set_option('display.max_colwidth', None)
# 2. 定义 CoT Prompt
prompt_v1 = """You are given a question in the form: 'Can a <Subject> be <Relationship> on a <Object>?'
For each question, analyze the physical and visual plausibility step by step.
Interpret this strictly as a statement about physical and commonsense plausibility in the real world.
Your task is to evaluate this question according to commonsense knowledge and visual plausibility.
For the given question, determine whether the situation is logically possible in the real world according to commonsense knowledge,
and whether this situation could realistically appear in a real photograph or video (i.e., no CGI, no fantasy, no metaphorical interpretations).
Ignore fictional, metaphorical, symbolic, microscopic, or extremely contrived edge cases unless explicitly stated.
Base your judgment on typical real-world conditions.
End your response with 'Final Answer: Yes' or 'Final Answer: No'.
"""

shots_cot = [
    {"role": "user", "content": "Can an arm be flying in the sky?"},
    {"role": "assistant", "content": "Reasoning: 1) An arm is a biological part of a human body. 2) In the real world, arms cannot detach and fly independently. 3) This would only happen in CGI or fantasy. \nFinal Answer: No"},
    {"role": "user", "content": "Can a car be driving on a road?"},
    {"role": "assistant", "content": "Reasoning: 1) Cars are vehicles designed for roads. 2) This is a standard real-world activity. 3) It is visually plausible. \nFinal Answer: Yes"},
    {"role": "user", "content": "Can a house be in a car?"},
    {"role": "assistant", "content": "Reasoning: 1) A house is significantly larger than a car. 2) A car cannot physically contain a house. 3) This violates physical scale and spatial logic. \nFinal Answer: No"},

]

# 3. 运行测试（复用你的 run_ablation 函数，但传入新的采样参数）
# 注意：原 run_ablation 内部使用了 sampling_params，我们需要临时替换它
import copy
original_params = sampling_params
sampling_params = sampling_params_cot

print("=== 实验 4: original prompt but cot ===")
df_res2 = run_ablation(prompt_v1, shots_cot)
display(df_res2)

# 还原参数
sampling_params = original_params

=== 实验 4: original prompt but cot ===

=> 测试完成! 在 50个样本上的准确率为: 86.0%



,Question,Expected,Model_Output,Correct
0,Can a car be on a street?,Yes,Reasoning: 1) Streets are designed for vehicles. 2) Cars are designed for streets. 3) This is a common and visually plausible scenario. \nFinal Answer: Yes,True
1,Can a person be wearing a hat?,Yes,Reasoning: 1) People wear hats as part of their clothing. 2) Hats are designed to be worn on the head. 3) This is a common and visually plausible scenario. \nFinal Answer: Yes,True
2,Can a boat be flying in a room?,No,"Reasoning: 1) Boats are designed for water, not air. 2) They do not have the physical capabilities to fly. 3) This would require defying the laws of physics and buoyancy. \nFinal Answer: No",True
3,Can a cat be sitting on a table?,Yes,Reasoning: 1) Cats are common household pets. 2) Tables are common household furniture. 3) It is a common and visually plausible scenario for a cat to be sitting on a table. \nFinal Answer: Yes,True
4,Can a pizza be eating a man?,No,"Reasoning: 1) A pizza is a food item, and it cannot consume or eat anything. 2) The concept of a pizza eating a man is biologically and physically impossible. 3) This is not visually plausible in the real world.\nFinal Answer: No",True
5,Can a train be parked on a track?,Yes,"Reasoning: 1) Trains are designed to run on tracks. 2) Parking a train on a track is a common real-world scenario, especially when not in use. 3) It is visually plausible.\nFinal Answer: Yes",True
6,Can a tree be riding a bike?,No,Reasoning: 1) Trees are stationary objects and do not have the ability to move on their own. 2) Bicycles require a rider to operate. 3) Trees do not possess the physical attributes or cognitive abilities to ride a bike. \nFinal Answer: No,True
7,Can a dog be holding a branch?,Yes,Reasoning: 1) Dogs are capable of holding objects in their mouths. 2) Branches are objects that can be grasped by a dog's mouth. 3) This is a common real-world scenario. \nFinal Answer: Yes,True
8,Can a flower be growing on a rock?,Yes,"Reasoning: 1) Flowers can grow in various environments, including rocky areas. 2) Some plants can grow on rocks using their roots or other adaptations. 3) This is a common occurrence in nature. \nFinal Answer: Yes",True
9,Can a logo be of a banana?,Yes,Reasoning: 1) A logo is a visual representation. 2) A banana is a real-world object that can be depicted in a logo. 3) This is a common and visually plausible scenario. \nFinal Answer: Yes,True
